In [1]:
import os

In [2]:
%pwd

'e:\\ml projects\\fraud-transaction-detection\\research'

In [3]:
os.chdir("../")

In [4]:
from dataclasses import dataclass 
from pathlib import Path 

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_url: str
    local_data_file:Path
    unzip_dir: Path

In [5]:
from src.transaction_fraud_detection.logging import logger
from src.transaction_fraud_detection.utils.common import read_yaml,get_size,create_directories
from src.transaction_fraud_detection.constants import *

In [6]:
class ConfigManager:
    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH,
        params_file_path=PARAM_FILE_PATH):

        self.config=read_yaml(config_file_path)
        self.schema=read_yaml(schema_file_path)
        self.params=read_yaml(params_file_path)

        create_directories([self.config.root_dir])

    def get_data_ingestion_config(self)->DataIngestionConfig:
        config=self.config.data_ingestion

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir)
            
        return data_ingestion_config

In [7]:
import zipfile
from urllib.request import urlretrieve


In [8]:
class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename,header=urlretrieve(
                url=self.config.source_url,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} download sucessfully from following header:\n{header}")

        else:
            logger.info(f"file is alreday exists of size:{get_size(self.config.local_data_file)}")

    
    def extract_file(self):
        unzip_path=self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file,"r")as zipref:
            zipref.extractall(unzip_path)

In [9]:
try:
    config=ConfigManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_file()
except Exception as e:
    raise e

[2025-08-11 21:03:00,765 : INFO : common : yaml_fileconfig\config.yaml loaded sucessfully]
[2025-08-11 21:03:00,769 : INFO : common : yaml_fileschema.yaml loaded sucessfully]


[2025-08-11 21:03:00,771 : INFO : common : yaml_fileparams.yaml loaded sucessfully]


BoxKeyError: "'ConfigBox' object has no attribute 'root_dir'"